In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported")

✓ Libraries imported


In [2]:
# Load CIC-IDS 2017 Thursday data
df_cicids = pd.read_csv(
    '../data/raw/cicids/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
    low_memory=False
)

# Clean column names
df_cicids.columns = df_cicids.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('/', '_')

print(f"CIC-IDS shape: {df_cicids.shape}")
print(f"\nColumns ({len(df_cicids.columns)}):")
print(df_cicids.columns.tolist()[:10], "...")
print(f"\nLabel distribution:")
print(df_cicids['label'].value_counts())

CIC-IDS shape: (170366, 79)

Columns (79):
['destination_port', 'flow_duration', 'total_fwd_packets', 'total_backward_packets', 'total_length_of_fwd_packets', 'total_length_of_bwd_packets', 'fwd_packet_length_max', 'fwd_packet_length_min', 'fwd_packet_length_mean', 'fwd_packet_length_std'] ...

Label distribution:
label
BENIGN                        168186
Web Attack � Brute Force        1507
Web Attack � XSS                 652
Web Attack � Sql Injection        21
Name: count, dtype: int64


In [3]:
#53 CSIC HTTP features
CSIC_FEATURES = [
    'url_length', 'url_path_depth', 'url_num_dots', 'url_num_special', 'url_num_hyphens',
    'url_num_underscores', 'url_num_percent', 'url_num_equal', 'url_num_ampersand', 'url_entropy',
    'url_has_risky_ext', 'url_has_double_encoding', 'query_length', 'query_num_params', 'query_num_equals',
    'query_num_special', 'query_num_percent', 'query_entropy', 'query_has_sqli', 'query_has_xss',
    'query_has_traversal', 'query_has_encoding', 'query_is_empty', 'body_length', 'body_entropy',
    'body_num_params', 'body_num_special', 'body_num_percent', 'body_num_quotes', 'body_num_semicolons',
    'body_num_brackets', 'body_has_sqli', 'body_has_xss', 'body_has_traversal', 'body_has_encoding',
    'body_is_empty', 'method_get', 'method_post', 'method_put', 'method_suspicious', 'cookie_length',
    'cookie_has_sqli', 'cookie_has_xss', 'cookie_is_present', 'content_type_is_form', 'content_type_is_json',
    'content_type_is_none', 'connection_is_close', 'connection_keep_alive', 'post_no_content_type',
    'get_with_body', 'post_empty_body', 'content_length_mismatch'
]

print(f"Total CSIC features to create: {len(CSIC_FEATURES)}")
print(f"\nFeatures:")
for i, f in enumerate(CSIC_FEATURES, 1):
    print(f"  {i:2}. {f}")

Total CSIC features to create: 53

Features:
   1. url_length
   2. url_path_depth
   3. url_num_dots
   4. url_num_special
   5. url_num_hyphens
   6. url_num_underscores
   7. url_num_percent
   8. url_num_equal
   9. url_num_ampersand
  10. url_entropy
  11. url_has_risky_ext
  12. url_has_double_encoding
  13. query_length
  14. query_num_params
  15. query_num_equals
  16. query_num_special
  17. query_num_percent
  18. query_entropy
  19. query_has_sqli
  20. query_has_xss
  21. query_has_traversal
  22. query_has_encoding
  23. query_is_empty
  24. body_length
  25. body_entropy
  26. body_num_params
  27. body_num_special
  28. body_num_percent
  29. body_num_quotes
  30. body_num_semicolons
  31. body_num_brackets
  32. body_has_sqli
  33. body_has_xss
  34. body_has_traversal
  35. body_has_encoding
  36. body_is_empty
  37. method_get
  38. method_post
  39. method_put
  40. method_suspicious
  41. cookie_length
  42. cookie_has_sqli
  43. cookie_has_xss
  44. cookie_is_pres

## 3. Feature Mapping: CIC-IDS Network Features → CSIC HTTP Features

In [4]:
# Mapping logic: Transform 79 CIC-IDS features → 53 CSIC HTTP features
# Based on conceptual similarity and attack indicator correlation

def map_cicids_to_csic(df):
    """
    Map CIC-IDS network flow features to CSIC HTTP features.
    Uses feature similarity and attack pattern matching.
    """
    
    df_mapped = pd.DataFrame()
    
    # ===== URL Features =====
    # url_length: proxy with total forward packet length
    df_mapped['url_length'] = (df['total_length_of_fwd_packets'] / (df['total_fwd_packets'] + 1)).fillna(0)
    
    # url_path_depth: proxy with number of PSH flags (request segments)
    df_mapped['url_path_depth'] = df['fwd_psh_flags'].fillna(0)
    
    # url_num_dots: proxy with packet length variance (complexity indicator)
    df_mapped['url_num_dots'] = (df['packet_length_std'] / 100).clip(0, 10).fillna(0)
    
    # url_num_special: proxy with flow entropy
    df_mapped['url_num_special'] = (df['fwd_packet_length_std'] / 50).clip(0, 20).fillna(0)
    
    # url_num_hyphens: proxy with average packet size
    df_mapped['url_num_hyphens'] = (df['average_packet_size'] / 100).clip(0, 5).fillna(0)
    
    # url_num_underscores: proxy with header length
    df_mapped['url_num_underscores'] = (df['fwd_header_length'] / 100).clip(0, 3).fillna(0)
    
    # url_num_percent: proxy with data packet ratio
    df_mapped['url_num_percent'] = ((df['act_data_pkt_fwd'] / (df['total_fwd_packets'] + 1)) * 10).fillna(0)
    
    # url_num_equal: proxy with SYN flag count
    df_mapped['url_num_equal'] = df['syn_flag_count'].fillna(0)
    
    # url_num_ampersand: proxy with ACK flag count
    df_mapped['url_num_ampersand'] = (df['ack_flag_count'] / 10).clip(0, 5).fillna(0)
    
    # url_entropy: proxy with packet length variance
    df_mapped['url_entropy'] = (df['packet_length_variance'] / 1000).clip(0, 8).fillna(0)
    
    # url_has_risky_ext: proxy - 1 if high data rate (suspicious), 0 otherwise
    df_mapped['url_has_risky_ext'] = ((df['flow_bytes_s'] > 50000) | (df['fwd_psh_flags'] > 5)).astype(int)
    
    # url_has_double_encoding: proxy - 1 if unusual packet pattern
    df_mapped['url_has_double_encoding'] = ((df['packet_length_std'] > 100) & (df['bwd_packet_length_std'] > 100)).astype(int)
    
    # ===== Query Features =====
    # query_length: proxy with bwd packet length mean
    df_mapped['query_length'] = (df['bwd_packet_length_mean']).fillna(0)
    
    # query_num_params: proxy with total backward packets
    df_mapped['query_num_params'] = (df['total_backward_packets'] / 10).clip(0, 50).fillna(0)
    
    # query_num_equals: proxy with bwd PSH flags
    df_mapped['query_num_equals'] = df['bwd_psh_flags'].fillna(0)
    
    # query_num_special: proxy with FIN flag count
    df_mapped['query_num_special'] = df['fin_flag_count'].fillna(0)
    
    # query_num_percent: proxy with RST flag count (abnormal termination)
    df_mapped['query_num_percent'] = df['rst_flag_count'].fillna(0)
    
    # query_entropy: proxy with flow IAT std
    df_mapped['query_entropy'] = (df['flow_iat_std'] / 1000).clip(0, 10).fillna(0)
    
    # query_has_sqli: proxy - 1 if suspicious data rate pattern
    df_mapped['query_has_sqli'] = ((df['flow_bytes_s'] > 100000) | (df['fwd_packet_length_max'] > 500)).astype(int)
    
    # query_has_xss: proxy - 1 if unusual packet variance
    df_mapped['query_has_xss'] = ((df['fwd_packet_length_std'] > 150) | (df['total_fwd_packets'] > 500)).astype(int)
    
    # query_has_traversal: proxy - 1 if bidirectional asymmetry
    df_mapped['query_has_traversal'] = ((df['down_up_ratio'] > 2) | (df['down_up_ratio'] < 0.5)).astype(int)
    
    # query_has_encoding: proxy - 1 if high PSH flag count (encoded/compressed data)
    df_mapped['query_has_encoding'] = ((df['fwd_psh_flags'] > 3) | (df['bwd_psh_flags'] > 3)).astype(int)
    
    # query_is_empty: proxy - 1 if no backward traffic
    df_mapped['query_is_empty'] = (df['total_backward_packets'] == 0).astype(int)
    
    # ===== Body Features =====
    # body_length: proxy with total fwd packet length
    df_mapped['body_length'] = (df['total_length_of_fwd_packets']).fillna(0)
    
    # body_entropy: proxy with packet length std
    df_mapped['body_entropy'] = (df['packet_length_std'] / 100).clip(0, 8).fillna(0)
    
    # body_num_params: proxy with fwd packets
    df_mapped['body_num_params'] = (df['total_fwd_packets'] / 10).clip(0, 50).fillna(0)
    
    # body_num_special: proxy with URG flags
    df_mapped['body_num_special'] = (df['fwd_urg_flags'] + df['bwd_urg_flags']).fillna(0)
    
    # body_num_percent: proxy with CWE flag count
    df_mapped['body_num_percent'] = df['cwe_flag_count'].fillna(0)
    
    # body_num_quotes: proxy with min packet length
    df_mapped['body_num_quotes'] = (df['min_packet_length'] / 100).clip(0, 5).fillna(0)
    
    # body_num_semicolons: proxy with max packet length
    df_mapped['body_num_semicolons'] = (df['max_packet_length'] / 1000).clip(0, 15).fillna(0)
    
    # body_num_brackets: proxy with ACE flag count
    df_mapped['body_num_brackets'] = df['ece_flag_count'].fillna(0)
    
    # body_has_sqli: proxy - 1 if abnormal length patterns
    df_mapped['body_has_sqli'] = ((df['max_packet_length'] > 1000) | (df['fwd_packet_length_max'] > 800)).astype(int)
    
    # body_has_xss: proxy - 1 if high variance
    df_mapped['body_has_xss'] = ((df['fwd_packet_length_std'] > 200) | (df['bwd_packet_length_std'] > 200)).astype(int)
    
    # body_has_traversal: proxy - 1 if unusual ratios
    df_mapped['body_has_traversal'] = ((df['avg_fwd_segment_size'] > avg_fwd_size * 1.5) if 'avg_fwd_segment_size' in df.columns else False).astype(int) if False else (df['total_backward_packets'] < 2).astype(int)
    
    # body_has_encoding: proxy - 1 if compression patterns
    df_mapped['body_has_encoding'] = ((df['fwd_packet_length_max'] > df['fwd_packet_length_mean'] * 3)).astype(int)
    
    # body_is_empty: proxy - 1 if no forward data packets
    df_mapped['body_is_empty'] = (df['act_data_pkt_fwd'] == 0).astype(int)
    
    # ===== Method Features =====
    # method_get: proxy - 1 if downstream dominated (GET = less upload)
    df_mapped['method_get'] = (df['down_up_ratio'] > 1.5).astype(int)
    
    # method_post: proxy - 1 if bidirectional (POST = request + response)
    df_mapped['method_post'] = ((df['total_fwd_packets'] > 0) & (df['total_backward_packets'] > 0)).astype(int)
    
    # method_put: proxy - 1 if high upload (PUT = upload data)
    df_mapped['method_put'] = (df['down_up_ratio'] < 1).astype(int)
    
    # method_suspicious: proxy - 1 if abnormal flags
    df_mapped['method_suspicious'] = ((df['rst_flag_count'] > 2) | (df['syn_flag_count'] > 5) | (df['fin_flag_count'] > 2)).astype(int)
    
    # ===== Cookie Features =====
    # cookie_length: proxy with init_win bytes forward
    df_mapped['cookie_length'] = (df['init_win_bytes_forward'] / 1000).clip(0, 65).fillna(0)
    
    # cookie_has_sqli: proxy - 1 if high data packet count
    df_mapped['cookie_has_sqli'] = (df['act_data_pkt_fwd'] > 50).astype(int)
    
    # cookie_has_xss: proxy - 1 if unusual window size
    df_mapped['cookie_has_xss'] = ((df['init_win_bytes_forward'] > 65535) | (df['init_win_bytes_backward'] > 65535)).astype(int)
    
    # cookie_is_present: proxy - 1 if bidirectional flow
    df_mapped['cookie_is_present'] = ((df['total_fwd_packets'] > 0) & (df['total_backward_packets'] > 0)).astype(int)
    
    # ===== Content-Type & Connection Features =====
    # content_type_is_form: proxy - 1 if POST-like (bidirectional)
    df_mapped['content_type_is_form'] = ((df['total_fwd_packets'] > 2) & (df['total_backward_packets'] > 2)).astype(int)
    
    # content_type_is_json: proxy - 1 if specific packet patterns
    df_mapped['content_type_is_json'] = ((df['fwd_packet_length_mean'] > 100) & (df['bwd_packet_length_mean'] > 100)).astype(int)
    
    # content_type_is_none: proxy - 1 if no data payload
    df_mapped['content_type_is_none'] = ((df['total_length_of_fwd_packets'] < 100) & (df['total_length_of_bwd_packets'] < 100)).astype(int)
    
    # connection_is_close: proxy - 1 if FIN flags present
    df_mapped['connection_is_close'] = (df['fin_flag_count'] > 0).astype(int)
    
    # connection_keep_alive: proxy - 1 if ACK dominated
    df_mapped['connection_keep_alive'] = (df['ack_flag_count'] > df['fin_flag_count']).astype(int)
    
    # ===== Anomaly Features =====
    # post_no_content_type: proxy - 1 if POST without typical response
    df_mapped['post_no_content_type'] = (((df['total_fwd_packets'] > df['total_backward_packets'] * 2)) & (df['bwd_packet_length_mean'] < 10)).astype(int)
    
    # get_with_body: proxy - 1 if GET with request body (unusual)
    df_mapped['get_with_body'] = ((df['total_length_of_fwd_packets'] > 500) & (df['total_backward_packets'] < 2)).astype(int)
    
    # post_empty_body: proxy - 1 if POST without payload
    df_mapped['post_empty_body'] = (((df['total_fwd_packets'] > 2) & (df['total_length_of_fwd_packets'] < 100)) | (df['fwd_packet_length_mean'] < 50)).astype(int)
    
    # content_length_mismatch: proxy - 1 if length anomaly
    df_mapped['content_length_mismatch'] = ((df['total_length_of_fwd_packets'] != df['fwd_packet_length_mean'] * df['total_fwd_packets'])).astype(int)
    
    return df_mapped

print(" Feature mapping function defined")

✓ Feature mapping function defined


## 4. Apply Feature Mapping

In [5]:
print("Applying feature mapping to CIC-IDS data...")
X_mapped = map_cicids_to_csic(df_cicids)

print(f"\n Feature mapping complete")
print(f"Shape: {X_mapped.shape}")
print(f"\nMapped features:")
print(X_mapped.columns.tolist())
print(f"\nData types:")
print(X_mapped.dtypes)
print(f"\nFirst few rows:")
print(X_mapped.head())

Applying feature mapping to CIC-IDS data...

✓ Feature mapping complete
Shape: (170366, 53)

Mapped features:
['url_length', 'url_path_depth', 'url_num_dots', 'url_num_special', 'url_num_hyphens', 'url_num_underscores', 'url_num_percent', 'url_num_equal', 'url_num_ampersand', 'url_entropy', 'url_has_risky_ext', 'url_has_double_encoding', 'query_length', 'query_num_params', 'query_num_equals', 'query_num_special', 'query_num_percent', 'query_entropy', 'query_has_sqli', 'query_has_xss', 'query_has_traversal', 'query_has_encoding', 'query_is_empty', 'body_length', 'body_entropy', 'body_num_params', 'body_num_special', 'body_num_percent', 'body_num_quotes', 'body_num_semicolons', 'body_num_brackets', 'body_has_sqli', 'body_has_xss', 'body_has_traversal', 'body_has_encoding', 'body_is_empty', 'method_get', 'method_post', 'method_put', 'method_suspicious', 'cookie_length', 'cookie_has_sqli', 'cookie_has_xss', 'cookie_is_present', 'content_type_is_form', 'content_type_is_json', 'content_type_

## 5. Prepare Labels (Binary: 0=Normal, 1=Attack)

In [6]:
# Create binary labels: 0 = Normal/BENIGN, 1 = Attack (any)
y = (df_cicids['label'].str.lower() != 'benign').astype(int)

print(f"Label distribution:")
print(f"  Normal (0): {(y == 0).sum():,}")
print(f"  Attack (1): {(y == 1).sum():,}")
print(f"\n  Attack rate: {y.mean()*100:.2f}%")
print(f"  Imbalance ratio: {(y == 0).sum() / (y == 1).sum():.1f}:1")

# Reset index
X_mapped = X_mapped.reset_index(drop=True)
y = y.reset_index(drop=True)

Label distribution:
  Normal (0): 168,186
  Attack (1): 2,180

  Attack rate: 1.28%
  Imbalance ratio: 77.1:1


## 6. Handle Missing/Infinite Values

In [7]:
# Replace inf/-inf with NaN
X_mapped = X_mapped.replace([np.inf, -np.inf], np.nan)

# Check for NaN
nan_cols = X_mapped.columns[X_mapped.isnull().any()].tolist()
print(f"Columns with NaN: {len(nan_cols)}")
if nan_cols:
    print(f"  {nan_cols}")
    print(f"\nFilling NaN with column medians...")
    for col in nan_cols:
        X_mapped[col] = X_mapped[col].fillna(X_mapped[col].median())

print(f"\n NaN handling complete")
print(f"Remaining NaN: {X_mapped.isnull().sum().sum()}")

Columns with NaN: 0

✓ NaN handling complete
Remaining NaN: 0


## 7. Create Train/Validation/Test Split (70/15/15)

In [8]:
# First split: Train+Val (85%) vs Test (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X_mapped, y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

# Second split: Train (70%) vs Val (15%)
split_ratio = 0.15 / 0.85  # 15% of 85%
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=split_ratio,
    random_state=42,
    stratify=y_temp
)

print("Train/Validation/Test Split (Stratified):")
print(f"\nTrain:  {X_train.shape[0]:,} samples (70%)")
print(f"  Normal: {(y_train == 0).sum():,}, Attack: {(y_train == 1).sum():,}")
print(f"\nVal:    {X_val.shape[0]:,} samples (15%)")
print(f"  Normal: {(y_val == 0).sum():,}, Attack: {(y_val == 1).sum():,}")
print(f"\nTest:   {X_test.shape[0]:,} samples (15%)")
print(f"  Normal: {(y_test == 0).sum():,}, Attack: {(y_test == 1).sum():,}")

Train/Validation/Test Split (Stratified):

Train:  119,256 samples (70%)
  Normal: 117,730, Attack: 1,526

Val:    25,555 samples (15%)
  Normal: 25,228, Attack: 327

Test:   25,555 samples (15%)
  Normal: 25,228, Attack: 327


## 8. Feature Scaling (Fit on Training Only)

In [9]:
# Fit scaler on training data ONLY
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print(f" Feature scaling complete (StandardScaler)")
print(f"\nScaled data shapes:")
print(f"  Train: {X_train_scaled.shape}")
print(f"  Val:   {X_val_scaled.shape}")
print(f"  Test:  {X_test_scaled.shape}")

✓ Feature scaling complete (StandardScaler)

Scaled data shapes:
  Train: (119256, 53)
  Val:   (25555, 53)
  Test:  (25555, 53)


## 9. Create Final Datasets with Labels

In [10]:
# Add labels back
train_final = X_train_scaled.copy()
train_final['label'] = y_train.values

val_final = X_val_scaled.copy()
val_final['label'] = y_val.values

test_final = X_test_scaled.copy()
test_final['label'] = y_test.values

print(f"Final datasets:")
print(f"  Train: {train_final.shape}")
print(f"  Val:   {val_final.shape}")
print(f"  Test:  {test_final.shape}")
print(f"\n All datasets ready")

Final datasets:
  Train: (119256, 54)
  Val:   (25555, 54)
  Test:  (25555, 54)

 All datasets ready


In [11]:
# Save to CSV
train_final.to_csv('../data/final/cicids_cv_train.csv', index=False)
val_final.to_csv('../data/final/cicids_cv_val.csv', index=False)
test_final.to_csv('../data/final/cicids_cv_test.csv', index=False)

print(" Datasets saved:")
print(f"  ../data/final/cicids_cv_train.csv ({train_final.shape})")
print(f"  ../data/final/cicids_cv_val.csv ({val_final.shape})")
print(f"  ../data/final/cicids_cv_test.csv ({test_final.shape})")

 Datasets saved:
  ../data/final/cicids_cv_train.csv ((119256, 54))
  ../data/final/cicids_cv_val.csv ((25555, 54))
  ../data/final/cicids_cv_test.csv ((25555, 54))
